# 03. Siniestralidad y conglomerados

In [61]:
# ------------------------------------------
# 1. Importación de librerías
# ------------------------------------------
from pathlib import Path
from itertools import combinations
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display, HTML, Markdown
from scipy.spatial.distance import cdist
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import AgglomerativeClustering, KMeans
from sklearn.metrics import (
    silhouette_score, silhouette_samples, calinski_harabasz_score,
    davies_bouldin_score, adjusted_rand_score
)

# ------------------------------------------
# 2. Configuración de rutas
# ------------------------------------------
ROOT = next((
    p for p in [Path.cwd(), *Path.cwd().parents]
    if (p / "data" / "processed" / "emision_ajustada_2025.parquet").exists()
), None)
if ROOT is None:
    raise FileNotFoundError("Coloca el notebook dentro del proyecto con data/processed.")
PROCESSED = ROOT / "data" / "processed"

# ------------------------------------------
# 3. Cargar las bases ajustadas
# ------------------------------------------
# Lectura directa para no depender de las librerías de otros análisis.
emision_ajustada = pd.read_parquet(PROCESSED / "emision_ajustada_2025.parquet")
siniestros_ajustados = pd.read_parquet(PROCESSED / "siniestros_ajustados_2025.parquet")
anios = list(range(2015, 2023))
columnas = [f"{i}_monto pagado_ajustada" for i in range(4)]

for base in [emision_ajustada, siniestros_ajustados]:
    base["año"] = pd.to_numeric(base["año"], errors="raise").astype(int)
    base["tipo_embarcacion"] = (
        base["tipo_embarcacion"].astype("string").str.strip().str.lower()
        .str.replace(r"\s+", " ", regex=True)
    )
    assert base[["año", "tipo_embarcacion"]].notna().all().all()
    assert base["tipo_embarcacion"].ne("").all()

# Los faltantes de desarrollos recientes no son pagos cero.
completos = siniestros_ajustados["año"].isin(anios)
assert np.isfinite(siniestros_ajustados.loc[completos, columnas]).all().all(),     "Hay pagos faltantes en 2015–2022: revisar antes de agrupar."
assert np.isfinite(emision_ajustada["prima_emitida_ajustada"]).all()
print(f"Emisión: {emision_ajustada.shape}")
print(f"Siniestros: {siniestros_ajustados.shape}")

Emisión: (21353, 11)
Siniestros: (2763, 14)


# 3.1. Preparación y cálculo de factores de siniestralidad

## 3.1.1. Factor de siniestralidad del mercado por año de origen

Calculamos la siniestralidad anual del mercado dividiendo los pagos observacos entre la prima emitida (factor de siniestralidad anual)

$$\text{Factor de siniestralidad}_{t}=\frac{\text{Siniestros del año de origen }t}{\text{Prima emitida del año }t}$$

In [62]:
# ------------------------------------------
# 3.1.1 Siniestralidad total por año
# ------------------------------------------

# Columnas de desarrollo ajustadas por INPC
columnas = [f"{i}_monto pagado_ajustada" for i in range(4)]

# Sumar los desarrollos de cada registro
siniestros_ajustados["siniestro_total"] = (
    siniestros_ajustados[columnas].sum(axis=1, min_count=1)
)

# Agrupar los siniestros por año de origen
siniestros_anual = (
    siniestros_ajustados
    .groupby("año")["siniestro_total"]
    .sum(min_count=1)
)

# Sumar todas las primas emitidas de cada año
primas_anual = (
    emision_ajustada
    .groupby("año")["prima_emitida_ajustada"]
    .sum(min_count=1)
)

# Calcular el factor de siniestralidad del mercado
# Las primas iguales a cero no constituyen un denominador válido.
factor_anual = siniestros_anual / primas_anual.where(primas_anual > 0)

factor_anual.reset_index().rename(
    columns={0: "factor_siniestralidad_anual"}
).set_index("año")

,factor_siniestralidad_anual
año,
2015,1.728247
2016,0.553433
2017,0.336023
2018,0.376379
2019,0.382350
2020,0.638191
2021,0.519178
2022,0.672990
2023,0.442951


In [63]:
# ------------------------------------------
# Métricas de siniestralidad por año
# ------------------------------------------

print("Métricas de siniestralidad anual")
factor_anual.describe()

Métricas de siniestralidad anual


count    11.000000
mean      0.611773
std       0.395429
min       0.336023
25%       0.379365
50%       0.519178
75%       0.655591
max       1.728247
dtype: float64

In [64]:
# ------------------------------------------------------
# Evolución del factor de siniestralidad anual por año
# ------------------------------------------------------

fig = px.line(
    factor_anual,
    x=factor_anual.index,
    y=factor_anual.values,
    markers=True,
    title="Evolución del factor de siniestralidad anual",
    labels={"x": "Año", "y": "Factor de siniestralidad"},
)

fig.update_traces(
    line=dict(width=2, color="#636EFA"),
    marker=dict(size=6)
)

fig.update_layout(
    template="plotly",

    # Fondo
    paper_bgcolor="#F2F2F2",
    plot_bgcolor="#F2F2F2",

    # Tipografía
    font=dict(
        family="Arial",
        size=14,
        color="#2A4163"
    ),

    # Título
    title=dict(
        x=0.03,
        y=0.95,
        font=dict(size=20)
    ),

    # Espacios
    margin=dict(
        l=65,
        r=40,
        b=65,
        t=130
    ),

    # Interacción
    hovermode="x unified",
    showlegend=False,

    # Dimensiones
    height=550
)

# Eje X
fig.update_xaxes(
    showgrid=False,
    zeroline=False,
    showline=False
)

# Eje Y
fig.update_yaxes(
    showgrid=True,
    gridcolor="white",
    gridwidth=1,
    zeroline=False,
    showline=False
)

fig.add_vrect(
    x0=2022.5, x1=2025.5, fillcolor="#EF9B38", opacity=0.10,
    line_width=0, annotation_text="Desarrollo incompleto", annotation_position="top left"
)
fig.show()

Entre los años 2015 y 2022 podemos observar que los factores de siniestralidad varian, por lo que se revisa por tipo de embarcación

## 3.1.2. Factor de siniestralidad por tipo de embarcación

In [65]:
# ------------------------------------------
# 3.1.2 Siniestralidad por tipo de embarcación
# ------------------------------------------

# Agrupar los siniestros por año de origen y tipo de embarcación
siniestros_tipo = (
    siniestros_ajustados
    .groupby(["año", "tipo_embarcacion"])["siniestro_total"]
    .sum(min_count=1)
)

# Sumar las primas emitidas por año y tipo de embarcación
primas_anual_tipo = (
    emision_ajustada
    .groupby(["año", "tipo_embarcacion"])["prima_emitida_ajustada"]
    .sum(min_count=1)
)

# Calcular el factor por tipo de embarcación
# Si hay prima positiva y no hay siniestros registrados, el monto es cero.
# Con prima nula o negativa, el factor no se calcula.
factor_anual_tipo = (
    siniestros_tipo.reindex(primas_anual_tipo.index, fill_value=0)
    / primas_anual_tipo.where(primas_anual_tipo > 0)
)

print("Factor de siniestralidad anual por tipo de embarcación")

display(
    factor_anual_tipo.rename("factor_siniestralidad_tipo")
    .reset_index()
    .set_index(["año", "tipo_embarcacion"])
)

Factor de siniestralidad anual por tipo de embarcación


factor_siniestralidad_tipo
año  tipo_embarcacion                                   
2015 abastecedor                                0.233185
     artefactos navales                              NaN
     atunero                                    0.045858
     barcaza o chalan                           0.111802
     barco-escuela                              0.000000
...                                                  ...
2025 remolcador                                 0.477299
     sardinero o anchovetero                    0.111391
     tiburoneros                                0.000000
     velero                                     0.693984
     yate                                       1.604225

[286 rows x 1 columns]

In [66]:
# Revisar pagos que no tienen una prima positiva asociada por tipo y año
revision_cruce = pd.concat([
    primas_anual_tipo.rename("Prima"), siniestros_tipo.rename("Pagos")
], axis=1)
revision_cruce = revision_cruce.loc[
    revision_cruce.index.get_level_values("año").isin(anios)
    & revision_cruce["Pagos"].fillna(0).ne(0)
    & ~revision_cruce["Prima"].gt(0)
]
print("Combinaciones con pagos y sin prima positiva:", len(revision_cruce))
display(revision_cruce)

Combinaciones con pagos y sin prima positiva: 9


Prima         Pagos
año  tipo_embarcacion                                   
2018 jet sky o moto acuatica -1.508781e+06  1.820318e+08
2015 no disponible                     NaN  1.378099e+07
2016 no disponible                     NaN  7.909986e+07
2017 no disponible                     NaN  3.910556e+07
2018 no disponible                     NaN  1.026613e+06
2019 artefactos navales                NaN  3.351274e+05
     no disponible                     NaN  2.995052e+06
2020 no disponible                     NaN  7.319878e+06
2021 no disponible                     NaN  7.267464e+07

Los pagos sin prima positiva permanecen en el total del mercado pero estos no producen un factor de siniestralidad válido por tipo y año, por lo que se revisa mas adelante 

## 3.1.3. Selección de los tipos de embarcación comparables

Se usa unicamente el periodo de 8 años de 2015 a 2022 porque estos años presentan los 4 años de desarrollo completos

In [67]:
# ------------------------------------------
# 3.1.3 Identificación de tipos de embarcación
# ------------------------------------------

# Convertir las primas agrupadas en un DataFrame
emision_tipos = primas_anual_tipo.reset_index()

# Identificar los tipos de embarcación en la base
tipos = emision_tipos[["tipo_embarcacion"]].drop_duplicates()
print("Número de tipos de embarcación:", len(tipos))

# --------------------------------------------------
# Cobertura por tipo durante 2015-2022
# --------------------------------------------------

# Seleccionar los años con desarrollo completo
periodo = emision_tipos[
    emision_tipos["año"].between(2015, 2022)
].copy()

# Contar años con prima positiva por tipo
cobertura_8 = (
    periodo
    .assign(positiva=periodo["prima_emitida_ajustada"] > 0)
    .groupby("tipo_embarcacion")["positiva"]
    .sum()
)

# Identificar tipos con prima positiva en los ocho años
tipos_completos = cobertura_8[cobertura_8 == 8].index

print("Tipos completos 2015-2022:", len(tipos_completos))

# --------------------------------------------------
# Participación de los tipos seleccionados
# --------------------------------------------------

seleccion = periodo["tipo_embarcacion"].isin(tipos_completos)

prima_mercado = periodo["prima_emitida_ajustada"].sum()
prima_tipos = periodo.loc[
    seleccion, "prima_emitida_ajustada"
].sum()

participacion = (
    prima_tipos / prima_mercado * 100
    if prima_mercado != 0 else np.nan
)

print(f"Participación de los tipos completos: {participacion:.2f}%")

Número de tipos de embarcación: 32
Tipos completos 2015-2022: 19
Participación de los tipos completos: 96.29%


Se seleccionan 19 tipos con prima positiva en los 8 años, lo cual equivale al 96.29% de la prima neta emitida en el periodo

In [68]:
# --------------------------------------------------
# 3.1.3. Tipos excluidos y motivo de su exclusión
# --------------------------------------------------

# Una fila por tipo y una columna por año de origen
primas_matriz = (
    primas_anual_tipo
    .unstack("año")
    .reindex(columns=range(2015, 2023))
)

# Resumir la situación de cada tipo en los ocho años
detalle_tipos = pd.DataFrame({
    "Años positivos": primas_matriz.gt(0).sum(axis=1),
    "Años en cero": primas_matriz.eq(0).sum(axis=1),
    "Años negativos": primas_matriz.lt(0).sum(axis=1),
    "Años sin registro": primas_matriz.isna().sum(axis=1)
})

# Identificar los años exactos que impiden tener trayectoria completa
anios = list(range(2015, 2023))
detalle_tipos["Años sin prima positiva"] = primas_matriz.apply(
    lambda fila: ", ".join(
        str(anio) for anio in anios
        if pd.isna(fila[anio]) or fila[anio] <= 0
    ),
    axis=1
)

# Describir la causa observada, sin asumir que es un error
# (un tipo puede presentar más de una situación)
def explicar_exclusion(fila):
    motivos = []
    if fila["Años sin registro"] > 0:
        motivos.append("Años sin registro de emisión")
    if fila["Años en cero"] > 0:
        motivos.append("Prima anual igual a cero")
    if fila["Años negativos"] > 0:
        motivos.append("Prima anual negativa")
    return "; ".join(motivos)

detalle_tipos["Motivo"] = detalle_tipos.apply(
    explicar_exclusion, axis=1
)

# Mostrar los tipos que no cumplen el criterio de ocho años positivos
tipos_excluidos = detalle_tipos.loc[
    ~detalle_tipos.index.isin(tipos_completos)
].copy()

print("Tipos seleccionados:", len(tipos_completos))
print("Tipos excluidos de la matriz:", len(tipos_excluidos))

display(
    tipos_excluidos.reset_index()
    .sort_values(["Años positivos", "tipo_embarcacion"])
)

# Comprobación: los tipos seleccionados y excluidos cubren
# todos los tipos de la base de emisión, incluso sin registros en 2015–2022.
assert len(tipos_completos) + len(tipos_excluidos) == len(primas_matriz)

Tipos seleccionados: 19
Tipos excluidos de la matriz: 13


,tipo_embarcacion,Años positivos,Años en cero,Años negativos,Años sin registro,Años sin prima positiva,Motivo
3,buque frigorifico,0,0,1,7,"2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022",Años sin registro de emisión; Prima anual nega...
11,porta contenedor,1,0,0,7,"2015, 2016, 2017, 2018, 2020, 2021, 2022",Años sin registro de emisión
5,buque-tanque quimico,2,0,0,6,"2015, 2018, 2019, 2020, 2021, 2022",Años sin registro de emisión
7,carguero tipo roll on - roll off,2,0,0,6,"2015, 2016, 2017, 2018, 2019, 2020",Años sin registro de emisión
2,barco-hidrografico,3,0,0,5,"2015, 2017, 2020, 2021, 2022",Años sin registro de emisión
6,buque-tanque-minerales,3,0,0,5,"2015, 2016, 2017, 2021, 2022",Años sin registro de emisión
4,buque transportador de autos,4,0,2,2,"2016, 2019, 2021, 2022",Años sin registro de emisión; Prima anual nega...
12,tiburoneros,4,0,0,4,"2016, 2017, 2018, 2019",Años sin registro de emisión
0,artefactos navales,6,0,1,1,"2015, 2019",Años sin registro de emisión; Prima anual nega...
1,barco-escuela,6,0,1,1,"2016, 2022",Años sin registro de emisión; Prima anual nega...


## 3.1.4. Construcción de la matriz para el análisis multivariado

Se construye una matriz con una fila por tipo de embarcación y columna una columna por año (no entra no disponible)

In [69]:
# ---------------------------------------------------------------------
# DataFrame del factor de siniestralidad de los tipos seleccionados
# ---------------------------------------------------------------------

# Convertir los factores en DataFrame
df_factores = factor_anual_tipo.rename("factor_siniestralidad").reset_index()

# Identificar los tipos seleccionados
filtro = df_factores["tipo_embarcacion"].isin(tipos_completos)

# Filtrar los tipos y años de interés
df_tipos = df_factores.loc[
    filtro & df_factores["año"].between(2015, 2022)
].copy()

# Solo la ausencia de registro de siniestro se trató como cero en 3.1.2.
# Un importe faltante no se sustituye automáticamente.
assert df_tipos["factor_siniestralidad"].notna().all()

In [70]:
# ---------------------------------------------------------------
# Mover el factor de cada año a una columna separada
# ---------------------------------------------------------------

df_tipos_pivot = df_tipos.pivot(
    index="tipo_embarcacion",
    columns="año",
    values="factor_siniestralidad"
)

df_tipos_pivot = df_tipos_pivot.reindex(
    columns=range(2015, 2023)
)

df_tipos_pivot.rename(
    columns={i: f"fs_{i}" for i in range(2015, 2023)},
    inplace=True
)

print("Dimensiones de la matriz:", df_tipos_pivot.shape)
display(df_tipos_pivot)

Dimensiones de la matriz: (19, 8)


año,fs_2015,fs_2016,fs_2017,fs_2018,fs_2019,fs_2020,fs_2021,fs_2022
tipo_embarcacion,,,,,,,,
abastecedor,0.233185,0.377512,0.438483,0.348956,0.034605,0.560429,0.571040,0.551124
atunero,0.045858,0.267407,0.080078,0.371242,0.264100,2.607975,4.002534,3.446874
barcaza o chalan,0.111802,2.137235,0.000000,0.000000,0.380713,0.000000,2.237615,0.052041
buque-tanque,35.767174,1.057684,0.708277,3.470587,6.861467,1.761038,0.163708,0.065368
buque-tanque petrolero y productos derivados,0.000735,0.122723,0.360971,0.182118,0.655586,0.033952,2.038834,0.038514
camaronero,0.426566,1.046860,0.800984,0.041806,0.241447,0.488655,0.114397,0.144984
carguero a granel,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
carguero general,1.464266,0.197423,0.062055,0.082406,0.941033,0.447100,0.004004,0.145652
dragas,0.000000,12.235499,0.000000,5.259638,1.178765,0.568701,0.000000,0.000000


# 3.2 Preprocesamiento Multivariado

## 3.2.1. Formulación del problema

Queremos identificar tipos con experiencias de pago semejantes. Cada tipo es una observación y sus ocho factores describen su historia.

## 3.2.2 Construcción de características

- **Siniestralidad acumulada:** pagos totales entre primas totales de 2015–2022.
- **Variabilidad temporal:** desviación estándar de los ocho factores.
- **Recurrencia:** proporción de años con factor mayor que 1.
- **Tendencia:** pendiente anual del factor, positiva si tiende a subir y negativa si tiende a bajar.
- **Prima acumulada:** volumen económico observado, sirve para interpretar la experiencia y no entra en las distancias. No sustituye una medida de exposición como número de embarcaciones.

In [71]:
# --------------------------------------------------
# 3.2.2. Construcción de características
# --------------------------------------------------

anios = list(range(2015, 2023))

# Matriz de factores anuales
X = df_tipos_pivot.copy()

# Recuperar primas y siniestros por tipo y año
primas_matriz = (
    primas_anual_tipo
    .unstack("año")
    .reindex(index=X.index, columns=anios)
)

siniestros_matriz = (
    siniestros_tipo
    .unstack("año")
    .reindex(index=X.index, columns=anios)
    .fillna(0)
)

# --------------------------------------------------
# Construcción de características
# --------------------------------------------------

caracteristicas = pd.DataFrame(index=X.index)

# 1. Siniestralidad acumulada
caracteristicas["Siniestralidad acumulada"] = (
    siniestros_matriz.sum(axis=1)
    / primas_matriz.sum(axis=1)
)

# 2. Variabilidad temporal
caracteristicas["Variabilidad temporal"] = (
    X.std(axis=1, ddof=1)
)

# 3. Recurrencia de alta siniestralidad
caracteristicas["Recurrencia"] = (
    (X > 1).mean(axis=1)
)

# 4. Tendencia temporal (pendiente anual)
caracteristicas["Tendencia"] = X.apply(
    lambda fila: np.polyfit(
        range(8), fila.to_numpy(dtype=float), 1
    )[0],
    axis=1
)

# --------------------------------------------------
# Indicador auxiliar de experiencia
# --------------------------------------------------

experiencia = pd.DataFrame(index=X.index)

experiencia["Prima acumulada"] = (
    primas_matriz.sum(axis=1)
)

# --------------------------------------------------
# Resultados
# --------------------------------------------------

print("Matriz de características:")
display(caracteristicas.round(4))

print("Indicador auxiliar de experiencia:")
display(experiencia.round(2))

caracteristicas_experiencia = pd.concat(
    [caracteristicas, experiencia], axis=1
)

Matriz de características:


,Siniestralidad acumulada,Variabilidad temporal,Recurrencia,Tendencia
tipo_embarcacion,,,,
abastecedor,0.2774,0.1863,0.000,0.0386
atunero,0.8627,1.6745,0.375,0.5948
barcaza o chalan,0.2143,0.9791,0.250,0.0055
buque-tanque,7.0308,12.1448,0.625,-2.9504
buque-tanque petrolero y productos derivados,0.2045,0.6860,0.125,0.1112
camaronero,0.3754,0.3559,0.125,-0.0877
carguero a granel,0.0000,0.0000,0.000,0.0000
carguero general,0.2538,0.5213,0.125,-0.0974
dragas,1.6985,4.3556,0.375,-0.7566


Indicador auxiliar de experiencia:


,Prima acumulada
tipo_embarcacion,
abastecedor,7.288401e+09
atunero,4.672413e+08
barcaza o chalan,2.008418e+08
buque-tanque,6.430191e+08
buque-tanque petrolero y productos derivados,3.249467e+09
camaronero,1.822178e+08
carguero a granel,1.587132e+07
carguero general,5.463084e+08
dragas,3.400357e+07


## 3.2.3. Diagnóstico multivariado

In [72]:
# --------------------------------------------------
# Distribuciones de las características
# --------------------------------------------------

import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Colores fijos para cada característica
colores_caracteristicas = {
    "Siniestralidad acumulada": "#636EFA",
    "Variabilidad temporal": "#00BFA6",
    "Recurrencia": "#EF9B38",
    "Tendencia": "#AB63FA"
}

# Crear figura con cuatro filas independientes
fig = make_subplots(
    rows=4,
    cols=1,
    shared_xaxes=False,
    vertical_spacing=0.08
)

# Crear un boxplot horizontal por característica
for i, (columna, color) in enumerate(
    colores_caracteristicas.items(), start=1
):
    fig.add_trace(
        go.Box(
            x=caracteristicas[columna],
            orientation="h",
            name=columna,
            boxpoints="all",
            jitter=0.4,
            pointpos=-1.3,
            line=dict(color=color, width=2),
            marker=dict(color=color, size=7, opacity=0.6),
            fillcolor=color,
            opacity=0.65,
            showlegend=True,
            # Identificar cada tipo de embarcación
        customdata=caracteristicas.index.to_numpy().reshape(-1, 1),
        hoveron="points",
        hovertemplate="<b>Tipo:</b> %{customdata[0]}<br><b>Valor:</b> %{x:.4f}<extra></extra>"

        ),
        row=i,
        col=1
    )

# Diseño general
fig.update_layout(
    template="plotly",
    title=dict(
        text="Distribución de las características de siniestralidad",
        x=0.03,
        y=0.98,
        font=dict(size=20)
    ),
    paper_bgcolor="#F2F2F2",
    plot_bgcolor="#F2F2F2",
    font=dict(
        family="Arial",
        size=14,
        color="#2A4163"
    ),
    margin=dict(l=75, r=50, b=60, t=125),
    height=650,
    width=1000
)


# Escala y cuadrícula independiente para cada característica
fig.update_xaxes(
    matches=None,
    showgrid=True,
    gridcolor="white",
    gridwidth=1,
    zeroline=True,
    showline=True
)

fig.update_yaxes(
    showgrid=False,
    showticklabels=False,
    zeroline=True,
    title=None
)


fig.show()

In [73]:
# --------------------------------------------------
# Matrices de correlación: Pearson y Spearman
# --------------------------------------------------

from plotly.subplots import make_subplots
import plotly.graph_objects as go

# Calcular correlaciones
correlacion_pearson = caracteristicas_experiencia.corr(method="pearson")
correlacion_spearman = caracteristicas_experiencia.corr(method="spearman")

# Crear figura con dos matrices
fig = make_subplots(
    rows=1,
    cols=2,
    subplot_titles=["Pearson", "Spearman"],
    horizontal_spacing=0.15
)

# Agregar matrices
for i, matriz in enumerate(
    [correlacion_pearson, correlacion_spearman], start=1
):
    fig.add_trace(
        go.Heatmap(
            z=matriz.values,
            x=matriz.columns,
            y=matriz.index,
            text=matriz.values,
            texttemplate="%{text:.2f}",
            textfont=dict(size=14),
            coloraxis="coloraxis",
            xgap=3,
            ygap=3,
            hovertemplate=(
                "%{y} / %{x}<br>"
                "Correlación: %{z:.3f}"
                "<extra></extra>"
            )
        ),
        row=1,
        col=i
    )

# Diseño general
fig.update_layout(
    template="plotly",

    # Fondo
    paper_bgcolor="#F2F2F2",
    plot_bgcolor="#F2F2F2",

    # Tipografía
    font=dict(
        family="Arial",
        size=14,
        color="#2A4163"
    ),

    # Título
    title=dict(
        text="Correlaciones entre características de siniestralidad",
        x=0.03,
        y=0.97,
        font=dict(size=20)
    ),

    # Escala de color compartida
    coloraxis=dict(
        colorscale="PuOr",
        cmin=-1,
        cmax=1,
        cmid=0,
        colorbar=dict(
            title="Correlación",
            thickness=15,
            len=0.75
        )
    ),

    # Dimensiones y espacios
    width=1450,
    height=650,
    margin=dict(l=180, r=100, b=140, t=130),

    showlegend=False
)

# Ejes
fig.update_xaxes(
    tickangle=-35,
    showgrid=False,
    zeroline=False
)

fig.update_yaxes(
    autorange="reversed",
    showgrid=False,
    zeroline=False
)

# Subtítulos
fig.update_annotations(
    font=dict(size=17, color="#2A4163")
)

fig.show()

Se observa que los valores extremos influyen en Pearson mientras que en Spearman es más robusto a estos valores extremos, la diferencia entre ellos nos idica que si hay valores extremos los cuales influyan, mientras que una correlación alta en ambos sugiere que existe una información algo redundante.

Se conserva los extremos pero guardamos recurrencia y tendencia para interpretar los grupos por lo tanto solo se usará la siniestralidad acumulada y la variabilidad temporal para la construcción de los conglomerados

## 3.2.4. Transformación y estandarización

Para evitar que los factores con valores altos afecten mucho en las distancias se aplica una transformación logaritmica $L_{it}=\log(1+F_{it})$ a cada factor anual, esta transformación conserva los ceros y reduce el peso de los valores grandes.

Luego para la construcción de los conglomerados usaremos:

- **Nivel:** El promedo de los 8 valores transformados 

- **Variabilidad:** desviación estándar muestral de esos valores.

Después estandarizamos ambas variables, restando su media y dividiendo por su desviación estándar. Así tienen escalas comparables.

In [74]:
# --------------------------------------------------
# Características para construir los conglomerados
# --------------------------------------------------
X_log = np.log1p(df_tipos_pivot)
caracteristicas_cluster = pd.DataFrame(index=X_log.index)
caracteristicas_cluster["Nivel"] = X_log.mean(axis=1)
caracteristicas_cluster["Variabilidad"] = X_log.std(axis=1, ddof=1)

X_log_std = pd.DataFrame(
    StandardScaler().fit_transform(caracteristicas_cluster),
    index=caracteristicas_cluster.index,
    columns=caracteristicas_cluster.columns
)
display(caracteristicas_cluster.round(4))
print("Correlación de Spearman entre nivel y variabilidad:")
display(caracteristicas_cluster.corr(method="spearman").round(3))

,Nivel,Variabilidad
tipo_embarcacion,,
abastecedor,0.3203,0.1431
atunero,0.6618,0.6740
barcaza o chalan,0.3497,0.5109
buque-tanque,1.2065,1.1777
buque-tanque petrolero y productos derivados,0.2849,0.3740
camaronero,0.3198,0.2402
carguero a granel,0.0000,0.0000
carguero general,0.2993,0.3239
dragas,0.7058,0.9899


Correlación de Spearman entre nivel y variabilidad:


,Nivel,Variabilidad
Nivel,1.000,0.733
Variabilidad,0.733,1.000


In [75]:
fig = px.box(
    X_log_std.melt(ignore_index=False, var_name="Característica", value_name="Valor").reset_index(),
    x="Valor", y="Característica", color="Característica", points="all",
    hover_data=["tipo_embarcacion"],
    color_discrete_sequence=["#636EFA", "#00BFA6"],
    title="Nivel y variabilidad transformados y estandarizados"
)
fig.update_layout(
    template="plotly", paper_bgcolor="#F2F2F2", plot_bgcolor="#F2F2F2",
    font=dict(family="Arial", size=14, color="#2A4163"),
    title=dict(x=0.03), height=450, showlegend=False
)
fig.update_xaxes(gridcolor="white")
fig.show()

# 3.3 Conglomerados

## 3.3.1. Escenarios y métodos

- **A:** Con outliers 

- **B:** Sin outliers 

Probamos $k=2,3,4$ con los metodos de agrupación:
- **Ward** reduce el incremento de variación interna 
- **liga completa** usa la mayor distancia entre dos grupos
-**K-means** minimiza distancias cuadráticas a centroides
-**K-medoids** distancias a observaciones representativas.

Usamos distancia euclidiana y revisamos métricas junto con tamaños e interpretación.

## 3.3.2. Métricas de evaluación

- **Silhouette:** compara la distancia al propio grupo con la del grupo vecino más cercano.

$$s(i)=\frac{b(i)-a(i)}{\max\{a(i),b(i)\}},\qquad s(i)\in[-1,1].$$

$a(i)$ es la distancia media al propio grupo y $b(i)$ al vecino más cercano. Cerca de 1 indica separación, cerca de 0 indica frontera y negativo sugiere una asignación poco adecuada. Se reporta el promedio.

- **Calinski–Harabasz:** compara la dispersión entre grupos con la dispersión interna.

$$CH=\frac{B/(k-1)}{W/(n-k)}.$$

$B$ y $W$ son las sumas de cuadrados entre y dentro de los grupos, $n$ el número de tipos y $k$ el de grupos. Un valor mayor es preferible al comparar la misma matriz.

- **WCSS:** mide la dispersión de los puntos respecto al centro de su grupo.

$$WCSS=\sum_{g=1}^{k}\sum_{i\in C_g}\lVert x_i-\bar{x}_g\rVert^2.$$

$\bar{x}_g$ es el centroide del grupo. Menor significa más compactación, pero suele disminuir al aumentar $k$; no basta para elegirlo. En K-medoids se presenta como diagnóstico, no como su función objetivo.

- **Davies–Bouldin:** resume cuánto se parecen los grupos a su vecino más similar.

$$DB=\frac{1}{k}\sum_{g=1}^{k}\max_{h\ne g}\frac{S_g+S_h}{d(c_g,c_h)}.$$

$S_g$ es la distancia media al centroide $c_g$ y $d$ la distancia entre centroides. Menor indica grupos compactos y separados.

- **ARI, índice de Rand ajustado:** compara dos clasificaciones de **los mismos tipos**, descontando el acuerdo esperado por azar.

$$ARI=\frac{RI-E(RI)}{\max(RI)-E(RI)}.$$

$RI$ mide acuerdo entre pares de observaciones. ARI = 1 indica clasificaciones iguales, cerca de 0 un acuerdo similar al azar y negativo un acuerdo menor. No depende del nombre de los grupos ni demuestra capacidad predictiva.

**Tamaños:** revisamos cuántos tipos contiene cada grupo. Una métrica alta con un grupo de un solo tipo no basta para justificar una segmentación útil.

Referencias de las definiciones: [métricas de agrupación de scikit-learn](https://scikit-learn.org/stable/modules/clustering.html#clustering-performance-evaluation).

In [76]:
# --------------------------------------------------
# Preparar escenarios comparables en su procedimiento
# --------------------------------------------------
X_A = X_log_std.copy()
restantes = caracteristicas_cluster.drop(
    index=["buque-tanque", "otro tipo de embarcacion"]
)
X_B = pd.DataFrame(
    StandardScaler().fit_transform(restantes),
    index=restantes.index, columns=restantes.columns
)
escenarios = {"A": X_A, "B": X_B}
print(f"Escenario A: {len(X_A)} tipos")
print(f"Escenario B: {len(X_B)} tipos")

Escenario A: 19 tipos
Escenario B: 17 tipos


In [77]:
# --------------------------------------------------
# Funciones de agrupación y evaluación
# --------------------------------------------------
def agrupar(X, metodo, k):
    valores = np.asarray(X, dtype=float)
    if metodo == "Ward":
        return AgglomerativeClustering(n_clusters=k, linkage="ward").fit_predict(valores)
    if metodo == "Liga completa":
        return AgglomerativeClustering(n_clusters=k, linkage="complete").fit_predict(valores)
    if metodo == "K-means":
        return KMeans(n_clusters=k, random_state=42, n_init=50).fit_predict(valores)
    if metodo == "K-medoids":
        distancias = cdist(valores, valores)
        mejor_costo = np.inf
        for med in combinations(range(len(valores)), k):
            costo = distancias[:, med].min(axis=1).sum()
            if costo < mejor_costo:
                mejor_costo = costo
                mejores = med
        etiquetas = distancias[:, mejores].argmin(axis=1)
        # Cada medoide conserva su grupo, también si hay empates por duplicados.
        etiquetas[list(mejores)] = np.arange(k)
        return etiquetas
    raise ValueError("Método no reconocido")


def evaluar_modelo(X, metodo, k_max):
    filas = []
    valores = np.asarray(X)
    for k in range(2, k_max + 1):
        etiquetas = agrupar(X, metodo, k)
        tamanos = pd.Series(etiquetas).value_counts().sort_values(ascending=False)
        if len(tamanos) != k:
            continue
        wcss = 0
        for grupo in np.unique(etiquetas):
            puntos = valores[etiquetas == grupo]
            wcss += ((puntos - puntos.mean(axis=0)) ** 2).sum()
        filas.append({
            "k": k,
            "Silhouette": silhouette_score(X, etiquetas),
            "Calinski-Harabasz": calinski_harabasz_score(X, etiquetas),
            "Davies-Bouldin": davies_bouldin_score(X, etiquetas),
            "WCSS": wcss,
            "Tamaños": tamanos.tolist(),
            "Grupo mínimo": tamanos.min()
        })
    return pd.DataFrame(filas)

In [78]:
# --------------------------------------------------
# 3. Evaluar los cuatro métodos
# --------------------------------------------------

metodos = [
    "Ward",
    "Liga completa",
    "K-means",
    "K-medoids"
]

# Mismo rango de k, compatible con ambos escenarios
k_max = min(4, len(X_A) - 1, len(X_B) - 1)

resultados = {}

for metodo in metodos:

    print(f"\n====================================================================== {metodo.upper()} ======================================================================\n")

    # Evaluación de cada escenario
    tabla_A = evaluar_modelo(X_A, metodo, k_max)
    tabla_B = evaluar_modelo(X_B, metodo, k_max)

    # Guardar resultados para utilizarlos después
    resultados[(metodo, "A")] = tabla_A
    resultados[(metodo, "B")] = tabla_B

    # Mostrar tablas lado a lado
    display(HTML(f"""
    <style>
        .comparacion {{
            display: flex;
            gap: 20px;
            align-items: flex-start;
            width: 100%;
        }}
        .escenario {{
            flex: 1;
            min-width: 0;
            overflow-x: auto;
        }}
        .escenario h4 {{
            font-family: Arial;
            color: #2A4163;
            margin-bottom: 12px;
        }}
        .tabla-metrica {{
            border-collapse: collapse;
            width: 100%;
            font-family: Arial;
            font-size: 12px;
        }}
        .tabla-metrica th,
        .tabla-metrica td {{
            padding: 8px;
            border-bottom: 1px solid #ddd;
            text-align: center;
        }}
        .tabla-metrica th {{
            background-color: #E4EAF3;
            color: #2A4163;
        }}
        .tabla-metrica tr:hover {{
            background-color: #EDF1F8;
        }}
    </style>

    <div class="comparacion">

        <div class="escenario">
            <h4>Escenario A — Todos los tipos</h4>
            {tabla_A.round(3).to_html(index=False, classes="tabla-metrica")}
        </div>

        <div class="escenario">
            <h4>Escenario B — Sin casos de revisión individual</h4>
            {tabla_B.round(3).to_html(index=False, classes="tabla-metrica")}
        </div>

    </div>
    """))


====================================================================== WARD ======================================================================



k,Silhouette,Calinski-Harabasz,Davies-Bouldin,WCSS,Tamaños,Grupo mínimo
2,0.672,46.537,0.529,10.167,"[15, 4]",4
3,0.631,46.272,0.418,5.601,"[15, 2, 2]",2
4,0.423,52.029,0.669,3.332,"[9, 6, 2, 2]",2
k,Silhouette,Calinski-Harabasz,Davies-Bouldin,WCSS,Tamaños,Grupo mínimo
2,0.607,17.301,0.468,15.789,"[15, 2]",2
3,0.540,19.200,0.385,9.084,"[14, 2, 1]",1
4,0.388,20.613,0.712,5.906,"[10, 4, 2, 1]",1



====================================================================== LIGA COMPLETA ======================================================================



k,Silhouette,Calinski-Harabasz,Davies-Bouldin,WCSS,Tamaños,Grupo mínimo
2,0.672,46.537,0.529,10.167,"[15, 4]",4
3,0.631,46.272,0.418,5.601,"[15, 2, 2]",2
4,0.554,50.611,0.373,3.417,"[14, 2, 2, 1]",1
k,Silhouette,Calinski-Harabasz,Davies-Bouldin,WCSS,Tamaños,Grupo mínimo
2,0.607,17.301,0.468,15.789,"[15, 2]",2
3,0.540,19.200,0.385,9.084,"[14, 2, 1]",1
4,0.334,20.703,0.694,5.885,"[9, 5, 2, 1]",1



====================================================================== K-MEANS ======================================================================



k,Silhouette,Calinski-Harabasz,Davies-Bouldin,WCSS,Tamaños,Grupo mínimo
2,0.710,49.596,0.408,9.700,"[16, 3]",3
3,0.631,46.272,0.418,5.601,"[15, 2, 2]",2
4,0.423,52.029,0.669,3.332,"[9, 6, 2, 2]",2
k,Silhouette,Calinski-Harabasz,Davies-Bouldin,WCSS,Tamaños,Grupo mínimo
2,0.607,17.301,0.468,15.789,"[15, 2]",2
3,0.540,19.200,0.385,9.084,"[14, 2, 1]",1
4,0.333,21.381,0.645,5.730,"[9, 5, 2, 1]",1



====================================================================== K-MEDOIDS ======================================================================



k,Silhouette,Calinski-Harabasz,Davies-Bouldin,WCSS,Tamaños,Grupo mínimo
2,0.710,49.596,0.408,9.700,"[16, 3]",3
3,0.631,46.272,0.418,5.601,"[15, 2, 2]",2
4,0.359,46.541,0.636,3.686,"[10, 5, 2, 2]",2
k,Silhouette,Calinski-Harabasz,Davies-Bouldin,WCSS,Tamaños,Grupo mínimo
2,0.607,17.301,0.468,15.789,"[15, 2]",2
3,0.292,15.206,0.785,10.718,"[10, 5, 2]",2
4,0.333,21.381,0.645,5.730,"[9, 5, 2, 1]",1


Las métricas favorecieron distintas soluciones según lo que evalúan. Dos grupos destacaron por su separación promedio, mientras que algunas soluciones de cuatro grupos destacaron por su compactación o por la relación entre dispersión interna y separación. Tres grupos destacaron por el acuerdo entre métodos y la interpretación de su composición.

Seleccionamos Ward con tres grupos en el escenario A para describir la siniestralidad histórica de los 19 tipos. La elección se apoya en que los cuatro métodos coincidieron y en que la separación adicional tiene una interpretación clara. El escenario B queda como contraste de sensibilidad.

## 3.3.3. Selección y composición de los grupos

Tomamos **Ward con 3 grupos en A**:

- G1: menor nivel y menor variabilidad
- G2: medio nivel y media variabilidad
- G3: mayor nivel y mayor variabilidad

In [89]:
# --------------------------------------------------
# Clasificación principal en tres grupos
# --------------------------------------------------

# Tipos que anteriormente se revisaban individualmente
tipos_extremos = pd.Index([
    "buque-tanque",
    "otro tipo de embarcacion"
], name="tipo_embarcacion")

# Clasificación Ward con dos grupos para los tipos restantes
etiquetas = pd.Series(
    agrupar(X_B, "Ward", 2),
    index=X_B.index,
    name="Grupo Ward"
)

# Ordenar los grupos por nivel mediano
orden = (
    restantes.groupby(etiquetas)["Nivel"]
    .median()
    .sort_values()
    .index
)

mapa = {
    orden[0]: "G1 · menor nivel",
    orden[1]: "G2 · mayor nivel"
}

clasificacion = etiquetas.map(mapa)

# Agregar los tipos extremos como tercer grupo
clasificacion = clasificacion.reindex(caracteristicas_cluster.index)
clasificacion.loc[clasificacion.index.intersection(tipos_extremos)] = (
    "G3 · mayor nivel y variabilidad"
)

clasificacion.name = "Grupo"

perfil_final = (
    caracteristicas_cluster
    .join(caracteristicas_experiencia)
    .join(clasificacion)
)

assert perfil_final["Grupo"].notna().all()

print("Composición de la clasificación:")
display(clasificacion.value_counts().rename("Tipos").to_frame())

for grupo, tabla in perfil_final.groupby("Grupo"):
    print(grupo)
    display(tabla.drop(columns="Grupo").round(3))

Composición de la clasificación:


,Tipos
Grupo,
G1 · menor nivel,15
G2 · mayor nivel,2
G3 · mayor nivel y variabilidad,2


G1 · menor nivel


,Nivel,Variabilidad,Siniestralidad acumulada,Variabilidad temporal,Recurrencia,Tendencia,Prima acumulada
tipo_embarcacion,,,,,,,
abastecedor,0.320,0.143,0.277,0.186,0.000,0.039,7.288401e+09
barcaza o chalan,0.350,0.511,0.214,0.979,0.250,0.006,2.008418e+08
buque-tanque petrolero y productos derivados,0.285,0.374,0.204,0.686,0.125,0.111,3.249467e+09
camaronero,0.320,0.240,0.375,0.356,0.125,-0.088,1.822178e+08
carguero a granel,0.000,0.000,0.000,0.000,0.000,0.000,1.587132e+07
carguero general,0.299,0.324,0.254,0.521,0.125,-0.097,5.463084e+08
embarcacion de investigacion,0.360,0.347,0.410,0.571,0.250,-0.081,6.894439e+07
escamero,0.223,0.244,0.210,0.333,0.000,-0.056,5.729231e+06
ferry-transbordador,0.316,0.213,0.402,0.299,0.000,0.017,2.083029e+08


G2 · mayor nivel


,Nivel,Variabilidad,Siniestralidad acumulada,Variabilidad temporal,Recurrencia,Tendencia,Prima acumulada
tipo_embarcacion,,,,,,,
atunero,0.662,0.674,0.863,1.675,0.375,0.595,4.672413e+08
dragas,0.706,0.990,1.698,4.356,0.375,-0.757,3.400357e+07


G3 · mayor nivel y variabilidad


,Nivel,Variabilidad,Siniestralidad acumulada,Variabilidad temporal,Recurrencia,Tendencia,Prima acumulada
tipo_embarcacion,,,,,,,
buque-tanque,1.206,1.178,7.031,12.145,0.625,-2.950,6.430191e+08
otro tipo de embarcacion,1.241,1.269,0.730,9.734,0.500,0.781,1.644207e+09


In [90]:
# --------------------------------------------------
# Resumen de la experiencia por grupo
# --------------------------------------------------
resumen_grupos = perfil_final.groupby("Grupo").agg(
    Tipos=("Nivel", "size"),
    Nivel_mediano=("Nivel", "median"),
    Variabilidad_mediana=("Variabilidad", "median"),
    Recurrencia_mediana=("Recurrencia", "median"),
    Tendencia_mediana=("Tendencia", "median"),
    Prima_acumulada=("Prima acumulada", "sum")
)
pagos_tipo = siniestros_matriz.sum(axis=1)
resumen_grupos["Pagos acumulados"] = pagos_tipo.groupby(clasificacion).sum()
resumen_grupos["Factor agregado"] = (
    resumen_grupos["Pagos acumulados"] / resumen_grupos["Prima_acumulada"]
)
resumen_grupos["Prima del mercado (%)"] = (
    100 * resumen_grupos["Prima_acumulada"] / prima_mercado
)
siluetas = pd.Series(silhouette_samples(X_B, etiquetas), index=X_B.index)
resumen_grupos["Silhouette medio"] = siluetas.groupby(clasificacion).mean()
display(resumen_grupos.round(3))

,Tipos,Nivel_mediano,Variabilidad_mediana,Recurrencia_mediana,Tendencia_mediana,Prima_acumulada,Pagos acumulados,Factor agregado,Prima del mercado (%),Silhouette medio
Grupo,,,,,,,,,,
G1 · menor nivel,15,0.331,0.324,0.125,-0.010,1.644999e+10,5.928018e+09,0.360,82.338,0.615
G2 · mayor nivel,2,0.684,0.832,0.375,-0.081,5.012449e+08,4.608391e+08,0.919,2.509,0.549
G3 · mayor nivel y variabilidad,2,1.224,1.224,0.562,-1.084,2.287226e+09,5.721448e+09,2.501,11.448,NaN


In [91]:
# --------------------------------------------------
# Separación de los grupos y tipos que los integran
# --------------------------------------------------
fig = px.scatter(
    perfil_final.reset_index(), x="Nivel", y="Variabilidad", color="Grupo",
    hover_name="tipo_embarcacion",
    hover_data=["Recurrencia", "Siniestralidad acumulada", "Prima acumulada"],
    color_discrete_sequence=["#636EFA", "#00BFA6", "#EF9B38", "#AB63FA"],
    title="Conglomerados por tipo de embarcación, 2015–2022"
)
fig.update_traces(marker=dict(size=11, opacity=0.85))
fig.update_layout(
    template="plotly", paper_bgcolor="#F2F2F2", plot_bgcolor="#F2F2F2",
    font=dict(family="Arial", size=14, color="#2A4163"),
    title=dict(x=0.03), height=580,
    legend=dict(orientation="h", y=-0.25), margin=dict(b=150)
)
fig.update_xaxes(title="Nivel: media de log(1 + factor)", gridcolor="white")
fig.update_yaxes(title="Variabilidad de log(1 + factor)", gridcolor="white")
fig.show()

# 3.4. Robustez y estabilidad

## 3.4.1. Acuerdo entre métodos

Comparamos Ward con los otros metodos en A con 3 grupos

In [94]:
# --------------------------------------------------
# Acuerdo entre algoritmos y escenarios
# --------------------------------------------------
filas = []
for metodo in ["Liga completa", "K-means", "K-medoids"]:
    y = agrupar(X_A, metodo, 3)
    etiquetas_A = pd.Series(
        agrupar(X_A, "Ward", 3),
        index=X_A.index
    )

    y_A_metodo = pd.Series(y, index=X_A.index)

    filas.append({
        "Comparación": f"A k=3: Ward vs {metodo}",
        "Tipos comparados": len(X_A),
        "ARI": adjusted_rand_score(
            etiquetas_A.to_numpy(),
            y_A_metodo.to_numpy()
        )
    })

acuerdo = pd.DataFrame(filas)
display(acuerdo.round(3))

,Comparación,Tipos comparados,ARI
0,A k=3: Ward vs Liga completa,19,1.0
1,A k=3: Ward vs K-means,19,1.0
2,A k=3: Ward vs K-medoids,19,1.0


**Conclusión.** Los 4 metodos coinciden que los 3 grupos propuestos son robustos al metodo de agrupación

## 3.4.2. Estabilidad temporal

Omitimos un año a la vez para detectar años influyentes al comparar 2015-2018 con 2019-2022 

Se recalcula el nivel, la variabilidad en cada periodo. 

In [104]:
# --------------------------------------------------
# Recalcular una partición para cualquier bloque de años
# --------------------------------------------------
def particion_periodo(tabla):
    log_tabla = np.log1p(tabla)
    resumen = pd.DataFrame({
        "Nivel": log_tabla.mean(axis=1),
        "Variabilidad": log_tabla.std(axis=1, ddof=1)
    })
    Z = StandardScaler().fit_transform(resumen)
    return agrupar(Z, "Ward", 3)

factores_A = df_tipos_pivot.loc[X_A.index]
filas = []
for anio in anios:
    y = particion_periodo(factores_A.drop(columns=f"fs_{anio}"))
    etiquetas_ocho = particion_periodo(factores_A)

    filas.append({
        "Año omitido": anio,
        "ARI vs ocho años": adjusted_rand_score(etiquetas_ocho, y),
        "Tamaños": sorted(
            pd.Series(y).value_counts().tolist(),
            reverse=True
        )
    })
estabilidad = pd.DataFrame(filas)
display(estabilidad.round(3))

primera = particion_periodo(factores_A[[f"fs_{a}" for a in range(2015, 2019)]])
segunda = particion_periodo(factores_A[[f"fs_{a}" for a in range(2019, 2023)]])
ari_mitades = adjusted_rand_score(primera, segunda)
print(f"ARI entre 2015–2018 y 2019–2022: {ari_mitades:.3f}")
display(pd.crosstab(
    pd.Series(primera, name="Grupo 2015–2018"),
    pd.Series(segunda, name="Grupo 2019–2022")
))

comparacion_mitades = pd.DataFrame({"Periodo 1": ["2015–2018"], "Periodo 2": ["2019–2022"], "ARI": [ari_mitades]})

,Año omitido,ARI vs ocho años,Tamaños
0,2015,0.962,"[15, 3, 1]"
1,2016,0.521,"[12, 5, 2]"
2,2017,1.000,"[15, 2, 2]"
3,2018,1.000,"[15, 2, 2]"
4,2019,1.000,"[15, 2, 2]"
5,2020,0.433,"[11, 5, 3]"
6,2021,0.284,"[8, 8, 3]"
7,2022,0.284,"[8, 8, 3]"


ARI entre 2015–2018 y 2019–2022: 0.278


Grupo 2019–2022,0,1,2
Grupo 2015–2018,,,
0,1,1,1
1,11,0,0
2,4,1,0


In [105]:
fig = px.bar(
    estabilidad, x="Año omitido", y="ARI vs ocho años",
    title="Sensibilidad de los grupos al omitir un año", text_auto=".3f"
)
fig.update_traces(marker_color="#636EFA")
fig.update_layout(
    template="plotly", paper_bgcolor="#F2F2F2", plot_bgcolor="#F2F2F2",
    font=dict(family="Arial", size=14, color="#2A4163"),
    title=dict(x=0.03), height=450
)
fig.update_xaxes(dtick=1)
fig.update_yaxes(range=[min(-0.05, estabilidad["ARI vs ocho años"].min()-0.05), 1.1], gridcolor="white")
fig.show()